#### 基于Qwen2.5-0.5B 基础预训练模型和LoRA微调的的用户交易意图分析分类微调模型

##### 加载Qwen2.5-0.5B模型
 - 使用ModelScope对Transformers包的封装加载Transformers的AutoModel和AutoTokenizer;
 - 分别加载模型和分词器
##### 调整模型输出层
- 添加一个分类层
- 维度为896、in_features=896、out_features=2

In [ ]:
from modelscope import AutoModelForSequenceClassification, AutoTokenizer
import pandas as pd
model_path = "Qwen/Qwen2.5-0.5B"

device="cuda"

model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path, num_labels=2).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)
model

##### 模型测试
- 使用cuda0设备、token IDS、res_token_ids转PyTorch Tensor

In [ ]:
token_ids = tokenizer(["这个活动的后续流程是怎样的？"], return_tensors="pt", padding=True, truncation=True).to(device)
token_ids.input_ids[-1, :]
res_token_ids = model(**token_ids)
res_token_ids

In [ ]:
from datasets import load_dataset

dataset_path = {
    "train": "./trading_intent_classification_lora/lora_adapter/train.csv",
    "val": "./trading_intent_classification_lora/lora_adapter/val.csv",
    "test": "./trading_intent_classification_lora/lora_adapter/test.csv"
}

dataset = load_dataset("csv", data_files=dataset_path)

tokenized_datasets = dataset.map(
    function=lambda examples: tokenizer(examples['Text'], truncation=True, max_length=512, padding=True),
    batched=True,
    remove_columns=['Text']
)

tokenized_datasets = tokenized_datasets.rename_column("Label", "labels")

##### 初始化LoRA微调
预计微调500万参数，会自动冻结除target_modules以外的层
- 注意安装PEFT包（Powered By Transformers）；
- 初始化LoraConfig，包含任务类型（因果语言模型，即Transformers）；
- 设置rank以及缩放、设置暂退率为0.1同时配置需要融合LoRA层的原始模型层（应用到全部线性层）
- 创建LoRA Layer、LoRA With Linear、replace_linear_with_lora

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

model.train()

lora_conf = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,     # LoRA Alpha（参数a，微调初始阶段使用默认值）
    lora_dropout=0.0, # LoRA微调暂退率（LoRA微调初始阶段使用默认值）
    # target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "score"],
    target_modules=["q_proj"], # 目标模型（初始阶段选择核心Q/V+顶层2~4层FFN）
)

model=get_peft_model(model=model, peft_config=lora_conf) # 会自动冻结除target_modules以外的层
model.print_trainable_parameters()
model.score

##### 配置训练参数
配置的训练参数包含以下内容：
- 设置模型训练检查点、结果的保存路径，已经覆盖已存在的输出目录；
- 需要训练过程中执行训练和评估；
- 超参设置，包括迭代次数、学习率、训练和验证集的批次大小、权重衰减系数；
- 设置模型的执行策略，包括每次epoch结束后进行一次验证、每次epoch结束后保存检查点、检查点的默认保存策略为每次epoch结束后；
- 注意：如果需要打印Training Loss，需要配置logging_strategy=epoch/steps；

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score
import numpy as np
import torch

model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir="./trading_intent_classification_lora",       # 模型训练检查点、成果的保存路径
    overwrite_output_dir=True,                               # 覆盖已存在的输出目录
    learning_rate=1e-4,                                      # 学习率（初始微调阶段采用默认值）
    per_device_train_batch_size=20,                          # 训练集批次大小
    per_device_eval_batch_size=20,                           # 验证集批次大小
    num_train_epochs=6,                                      # 迭代次数
    # max_steps=120,                                          # 迭代次数（改用steps）
    # weight_decay=0.01,                                     # 权重衰减系数（暂不设置）
    logging_strategy="epoch",                                # 每个epoch结束后记录日志（如果配置为steps需要设置logging_steps参数）
    eval_strategy="epoch",                                   # 验证过程在每次epoch结束后执行
    save_strategy="epoch",                                   # 在每次epoch结束后保存checkpoint
    metric_for_best_model="accuracy",                        # 选择最佳模型的指标（准确率）
    greater_is_better=True,                                  # 评估指标越大越好（True）
    report_to="tensorboard",                                 # 启动TensorBoard
    run_name="trading_intent_classification",                # 运行名称
    logging_dir="./trading_intent_classification_lora/logs", # 日志目录
    load_best_model_at_end=False,
    logging_steps=5
)

def compute_metrics(eval_pred) -> dict[str, float]:
    """模型评估指标计算

    Args:
        eval_pred (_type_): 模型训练输出

    Returns:
        dict[str, float]: 模型指标字典
    """
    
    logits, labels = eval_pred
    predictions = np.argmax(a=logits, axis=-1)
    
    return {"accuracy": accuracy_score(y_true=labels, y_pred=predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt"),
    compute_metrics=compute_metrics,
    callbacks=[
        # EarlyStoppingCallback(
        #     early_stopping_patience=2    # 容忍2轮无改进
        # )
    ],
)

##### 开始训练模型
- 需要设置train的resume_from_checkpoint=True，使训练可自动加载ouput_dir中的最新检查点
- 注意：如果一开始没有检查点，则无法进行训练，需要先开启训练，后续有加载需要时再配置resume_from_checkpoint参数

##### TensorBoard
- 使用TensorBoard进行训练日志、数据可视化，是TensorFlow的可视化工具，与Transformers原生集成；
- 需要安装依赖（pip install tensorboard）；
- 在TrainingArguments中需要增加参数report_to="tensorboard"；

##### 超参调整
- train loss快速下降、val loss高 -> rank过大产生过拟合，减小rank；
- 高rank表示高自由度，能学习到样本级别噪声、prompt偏置、token位置偶然相关性，数据集中的特有模式等；
- 低rank则使模型只学最重要的方向；
- 一次只改变一个超参，先确认最重要的容量参数（rank+target module）是否合理，再微调其他超参；
- LoRA微调的初始阶段，先用默认值设置a、dropout、learning_rate，先验证rank（8~16）初始和target module的合理性（核心Q/V+顶层2~4层FFN），待训练曲线率、dropout或scaling进行微调；
- 训练观察阶段：小规模试验；

In [ ]:
trainer.train()

##### 模型测试
使用测试集中的数据测试模型的准确率

In [ ]:
test_dataset_df = pd.read_csv(filepath_or_buffer="./trading_intent_classification_lora/lora_adapter/test.csv")
tests_df = test_dataset_df.reset_index(drop=True)
tests_df

all_samples = len(tests_df)
correct_num = 0
model.eval()

for i in range(0, all_samples):
    token_ids = tokenizer([tests_df['Text'][i]], return_tensors="pt", padding=True, truncation=True).to(device)
    if accuracy_score(y_pred=torch.argmax(input=model(**token_ids).logits, dim=-1).to("cpu").numpy(), y_true=[tests_df['Label'][i]]) == 1.0:
        correct_num+=1
accuracy = f"Test Accuracy:{correct_num / all_samples}"
accuracy

In [ ]:
token_ids = tokenizer([
    "Text:线上评分过程公平吗？\nType:讲好中国故事\nTag:(总)复选未晋级 (总)复选未完赛 大连-3月25周三19.00大连-3月27周五19.00大连-3月28周六10.30大连-3月28周六14.30大连-3月28周六19.00大连-3月29周日10.30沈阳-3月27周五19.00沈阳-3月29周日10.30营口-3月26周四19.00营口-3月29周日16.00锦州-3月28周六16.00大连-3月29周日14.30沈阳-3月28周六10.30沈阳-3月29周日14.30营口-3月28周六9.00锦州-3月24周二19.00锦州-3月29周日9.00大连-3月29周日19.00沈阳-3月28周六14.30沈阳-3月29周日19.00营口-3月28周六16.00锦州-3月26周四19.00锦州-3月29周日16.00沈阳-3月25周三19.00沈阳-3月28周六19.00营口-3月24周二19.00营口-3月29周日9.00锦州-3月28周六9.00"
], return_tensors="pt", padding=True, truncation=True).to("cuda")
accuracy_score(y_pred=torch.argmax(input=model(**token_ids).logits, dim=-1).to("cpu").numpy(), y_true=[1])

##### 保存和加载模型
- 使用model.save_pretrained(save_path)保存模型
- 使用tokenizer.save_pretrained(save_path)保存对应的分词器
- 使用PeftModel.from_pretrained(base_model, save_path)加载并使用

In [ ]:
# 保存模型
save_directory = "./trading_intent_classification_lora/lora_adapter"
model.save_pretrained(save_directory=save_directory)

In [ ]:
# 模型上传
!modelscope upload "modelscope_mp_260903038/ckcz-trading-intent" "./trading_intent_classification_lora/lora_adapter" --token "ms-8c6ec26a-dcf8-4df7-984d-a5f30b5bd026"

In [ ]:
# 清理训练缓存
import gc

# 清理GPU缓存
torch.cuda.empty_cache()
gc.collect()

# 删除旧的模型和Trainer
del model
del trainer
del training_args
del lora_conf

# 再次清理GPU缓存
torch.cuda.empty_cache()
gc.collect()